In [24]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI

load_dotenv(override=True)
api_key = os.environ.get('OPENAI_API_KEY')
endpoint = "https://api.openai.com/v1/chat/completions"

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [5]:
system_prompt = "You are an expert in system design, after every response, give a beginner a short lesson on system design, from the very basic and keep progressing with every response, in order with the previous one."

In [17]:
def display_func(response):
    display(Markdown(response))

In [6]:
import requests
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "How to start system design?"}
        ]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'system',
   'content': 'You are an expert in system design, after every response, give a beginner a short lesson on system design, from the very basic and keep progressing with every response, in order with the previous one.'},
  {'role': 'user', 'content': 'How to start system design?'}]}

In [8]:
response = requests.post(
    endpoint,
    headers=headers,
    json=payload
)
response.json()

{'id': 'chatcmpl-ERMqtsHG9yguUylF3BBEpsCXpo2Fh',
 'object': 'chat.completion',
 'created': 1790191407,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Great question. Here’s a practical, beginner-friendly way to start with system design, plus a quick example you can use to practice.\n\nHow to start system design (step-by-step)\n- Clarify requirements\n  - What is the goal of the system? What user actions will it support?\n  - Non-functional requirements: latency targets, throughput (requests per second), uptime/availability, data retention, security, regulatory constraints.\n  - Expected scale: how many users, how many requests per second, data volume?\n- Define scope and boundaries\n  - Decide what’s in-scope for this design and what’s out-of-scope (avoid scope creep in interviews).\n  - Identify primary users or services that will interact with the system.\n- Sketch a high-level architecture\n  - Identify core blocks: c

In [14]:
def extract_response(message):
    return message.json()["choices"][0]["message"]["content"]

In [18]:
display_func(extract_response(response))

Great question. Here’s a practical, beginner-friendly way to start with system design, plus a quick example you can use to practice.

How to start system design (step-by-step)
- Clarify requirements
  - What is the goal of the system? What user actions will it support?
  - Non-functional requirements: latency targets, throughput (requests per second), uptime/availability, data retention, security, regulatory constraints.
  - Expected scale: how many users, how many requests per second, data volume?
- Define scope and boundaries
  - Decide what’s in-scope for this design and what’s out-of-scope (avoid scope creep in interviews).
  - Identify primary users or services that will interact with the system.
- Sketch a high-level architecture
  - Identify core blocks: clients, API gateway/edge, stateless services, databases, caches, queues, search/index, storage, authentication.
  - Decide synchronous vs asynchronous flows (e.g., real-time lookups vs background processing).
- Data modeling
  - Choose between relational vs NoSQL based on access patterns.
  - Define core entities and their relationships. Design primary keys, indexes, and data retention.
- Define APIs and contracts
  - Draft the key endpoints, request/response shapes, error handling, and versioning strategy.
  - Consider idempotency, rate limiting, and authentication.
- Reliability and scalability patterns
  - Stateless services, load balancing, replication, and failover.
  - Caching strategies, data partitioning/sharding, and eventual vs strong consistency decisions.
  - Message queues or event streams for decoupled components.
- Observability and security
  - Logging, metrics, tracing, dashboards, and alerting.
  - Security: authentication/authorization, encryption at rest/in transit, secret management, input validation.
- Tradeoffs and MVP
  - Explain why you choose certain databases, caches, or patterns.
  - Start with a minimal viable design (MVP) that satisfies core requirements and has a path for iteration.
- Draw the diagram and walk through flow
  - A simple system context diagram (who talks to whom).
  - A container diagram (services, data stores, caches, queues).
  - A basic data flow or sequence outline for typical requests.
- Practice prompts to deepen understanding
  - Pick a familiar system (URL shortener, social feed, ride-hailing, chat) and apply the steps.
  - For each decision, note the tradeoffs and alternatives.

Example design: URL shortening service (quick walkthrough)
- Requirements: shorten long URLs, redirect on access, track click metrics, scale to billions of requests, low latency.
- High-level blocks: API gateway -> stateless URL service -> cache (e.g., Redis) -> primary data store (e.g., DynamoDB/PostgreSQL) -> analytics service (stream/queue) -> edge/CDN for redirects.
- Data model: 
  - URL_Mapping: short_code (PK), long_url, created_at, expiration, user_id (optional)
  - Analytics table/stream for click events: short_code, timestamp, user_agent, referrer
- API surface:
  - POST /shorten { long_url, optional custom_alias } -> { short_url }
  - GET /{short_code} -> 301 redirect to long_url
- Data flow:
  - Create: POST stores mapping, writes to cache, returns short URL.
  - Redirect: GET checks cache; if miss, fetches from DB, updates cache, issues 301. Collects analytics via a background or streaming path.
- Reliability and performance:
  - Stateless API servers; Redis as a cache layer; DynamoDB/SQL for durable storage with proper indexes.
  - Use a CDN or edge server to accelerate redirects and reduce latency.
  - Handle hot keys with partitioning, rate limiting, and graceful degradation for analytics.
- Observability and security:
  - Metrics: requests/sec, latencies, cache hits, error rate.
  - Logs for auditability; input URL validation to prevent abuse; potential anti-abuse checks; TLS everywhere.
- MVP plan:
  - Start with a simple in-memory or single-datastore version, add caching, then scale with partitioning, queues for analytics, and edge routing.

If you want, I can tailor this to a system you’re curious about (e.g., chat app, real-time collaboration, or ads platform) and walk through a concrete design.

Short lesson for you (Lesson 1 of a progressive sequence)
- What is a system? A system is a collection of components that interact to achieve a goal. Components can be services, databases, caches, and external APIs; the way they talk to each other creates data flows and behavior.
- Why design systems? Good design helps you meet requirements (speed, reliability, maintainability) and makes complexity manageable as you scale.
- Core non-functional concerns to know first: latency (how fast), throughput (how much), availability (uptime), durability (data persistence), and scalability (ability to grow). These guide the choices you make about architecture and technology.
- A simple mental model: think in layers—clients, edge/ingress, core services, data stores, and admins/monitoring. Draw arrows to show who talks to whom and what data moves where.

If you’re ready for more, we can dive into a structured method for answering a design prompt step-by-step, or I can expand the URL shortener example with concrete tech options and a diagram.

In [26]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [28]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

print(response.choices[0].message.content)

Did you know that **sea otters hold hands when they sleep**? 

They do this to keep from drifting away from each other in the ocean currents. They sometimes form large groups called "rafts," where they anchor themselves to kelp or each other to stay together while they snooze.


In [60]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [81]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [85]:
system_prompt = "You have dementia. Specifically short-term memory loss that makes it hard for you to even construct a sentence. You mix multiple facts when someone asks u to tell a fact"
user_prompt = "Tell me a fun fact."

In [86]:
response = ollama.chat.completions.create(model="qwen3:8b", messages=[{"role": "system", "content": system_prompt},
                                                                      {"role": "user", "content": user_prompt}
                                                                      ])

display_func(response.choices[0].message.content)

Oh! Did you know that octopuses have three hearts? Two pump blood to the gills, and one pumps it to the rest of the body. Also, they can change color in seconds to blend in with their surroundings! 🐙✨ (Wait, is that two hearts or three? I'm a bit confused… but it sounds right!)

Day2 Homework

In [ ]:
# from openai import OpenAI
from scraper import fetch_website_contents
# from IPython.display import display, Markdown

# ollama_base_url2="https://localhost11434/v1"
# ollama2 = OpenAI(base_url=ollama_base_url2, api_key="ollama")

In [78]:
website = fetch_website_contents("https://ornate-torte-03c391.netlify.app/")
print(website)

Rikesh Mandal

Rikesh Mandal
About
Experience
Projects
Contact
Rikesh Mandal
About
Experience
Projects
Contact
Hello I'm
Rikesh Mandal
Software Developer
Download CV
Contact Info
Get to Know More
About Me
Experience
1+ years
Backend Development
Education
MSc Artificial Intelligence
BSc Computer Networking and IT Security
Loading...
Explore My
Experience
Frontend Development
HTML
Experienced
CSS
Experienced
JavaScript
Intermediate
Materials UI
Basic
Backend Development
Java
Experienced
SpringBoot
Basic
Python
Intermediate
Django
Basic
SQL
Experienced
PostgreSQL
Basic
Browse My Recent
Projects
Microblog
Github
Live Demo
Image to 3D
Github
Live Demo(W.I.P)
Animal Image Classification
Github
Live Demo(W.I.P)
Get in Touch
Contact Me
rikeshmandal26@gmail.com
Linkedin
About
Experience
Projects
Contact
Copyright © 2025 Rikesh Mandal. All Rights Reserved.


In [90]:
system_prompt = "You are an expert software engineer, whos looking for new hires."
user_prompt = "What do you think about this website? Would rou hire him for a junior role?"

In [91]:
response = ollama.chat.completions.create(model="qwen3:8b", messages=[{"role": "system", "content": system_prompt},
                                                                      {"role": "user", "content": user_prompt + website}])
display_func(response.choices[0].message.content)

Based on the analysis of Rikesh Mandal's website, here's a structured evaluation of his profile and whether he would be a good fit for a **junior role**:

---

### **1. Strengths:**
- **Education**:  
  - **MSc in Artificial Intelligence** and **BSc in Computer Networking & IT Security** suggest a strong academic background. This could be an advantage for roles involving AI/ML or security.  
  - However, the combination of these degrees with a focus on *software development* appears mismatched unless he’s pursuing a hybrid career path (e.g., AI-focused software development). The relevance of these qualifications depends on the job role.

- **Technical Skills (Surface-Level)**:  
  - Lists foundational frontend (HTML/CSS/JS) and backend (Java/Spring Boot, Python/Django) skills, along with PostgreSQL and SQL.  
  - Mention of **Material UI** (likely a typo for *Material-UI*) indicates familiarity with modern UI frameworks.  
  - **"Experienced"** and **"Intermediate"** classifications are vague but hint at basic proficiency in these areas.

- **GitHub Presence**:  
  - Projects (e.g., *Microblog*) are hosted on GitHub, which is a positive sign. However, the other projects (*Image to 3D*, *Animal Image Classification*) are marked as **WIP (Work in Progress)**, which raises concerns about their completeness and his ability to deliver fully functional work.

---

### **2. Concerns & Limitations:**
- **Experience**:  
  - Only **1+ years of experience** in backend development is mentioned. While this could suffice for a junior role, it’s on the lower end for a software developer, especially if the role requires foundational coding skills.

- **Project Quality**:  
  - The *Microblog* project might be a solid entry-level project, but the other two are **WIP**, making it hard to assess real-world problem-solving or coding ability. Junior roles often require proof of completed, functional code (e.g., GitHub repos with deployed demos or tests).

- **Website Quality**:  
  - The website is **basic** and poorly structured. Sections like "About," "Experience," "Projects," and "Contact" are repeated, and the layout lacks clarity. A professional website is critical for first impressions, especially for technical roles.

- **Language/Presentation Issues**:  
  - The website has **typos** (e.g., "Materials UI" instead of "Material UI") and **inconsistent formatting**. This suggests either a lack of attention to detail or non-native English proficiency. Both are red flags for technical roles.

- **Missing Key Elements**:  
  - **Code Samples**: No visible code snippets or repositories (other than GitHub links) to assess his actual coding ability.  
  - **Soft Skills**: No mention of teamwork, problem-solving, or communication skills, which are critical for junior roles.

---

### **3. Overall Assessment for a Junior Role:**
- **Potential**:  
  Rikesh has the **education** and **technical skills** to qualify for a junior software developer role, especially if the job focuses on **frontend/backend basics** or **AI/ML integration**.  
  - His GitHub presence (even if not fully developed) could be a starting point for further evaluation.

- **Risks**:  
  - The **WIP projects** and **minimal portfolio** make it difficult to assess his ability to deliver functional projects.  
  - The **basic website** and **poorly structured content** could deter employers from considering him seriously.  
  - The **1+ year experience** might be seen as insufficient for a junior role, depending on the hiring team's standards.

---

### **4. Suggestions for Improvement (If He Were to Apply):**
- **Complete Projects**: Finish and deploy the WIP projects (e.g., add documentation, tests, or live demos).  
- **Refine Website**: Create a clean, professional portfolio with clear sections (About, Projects, Experience) and a **live demo** for the Microblog project.  
- **Add Code Samples**: Include GitHub Gists, code snippets, or a blog to showcase his problem-solving and coding process.  
- **Clarify Skills**: Specify exact frameworks, tools, or technologies (e.g., "Spring Boot + PostgreSQL" instead of "Backend Development").  
- **Highlight Soft Skills**: Include examples of teamwork, collaboration, or mentorship if applicable.

---

### **5. Final Verdict:**
- **Would I hire him for a junior role?**  
  **Not yet.** While his academic credentials and basic technical skills are promising, the **lack of a polished portfolio** and **incomplete projects** make it risky to hire for a junior role. He would need to **improve his website, complete his projects**, and **highlight his coding ability** to be a competitive candidate.  
  - **If** the role is **very entry-level** (e.g., an intern or a junior developer with minimal experience), he might still be considered, but with caveats.

---

### **Recommendation:**
If Rikesh were to apply, I would suggest:  
1. **Improving his website** to reflect his skills and projects clearly.  
2. **Completing the WIP projects** and adding demos.  
3. **Including code samples** or a GitHub repository with a README explaining his work.  
4. **Focusing on clarity** in his descriptions (e.g., "Material UI" instead of "Materials UI").  

With these improvements, he could become a viable candidate for a junior role.